# Evaluation fuer den Kursbericht
Reihenfolge: 2 und 3 (Agent mit/ohne Selbstkorrektur) und 4 (Baseline) kosten API-Guthaben, alles andere nicht.
**Waehrend 2-4 laufen, keine anderen Agenten starten** - die Kosten werden ueber den OpenRouter-Verbrauch gemessen.
Ergebnis am Ende: `eval/summary.md` und `eval/summary_chart.png`.

In [ ]:
# 1) Setup: Skripte laufen als eigener Prozess aus dem FinRobot-Ordner
import subprocess, sys
from pathlib import Path
ROOT = Path.cwd().resolve().parents[1]      # ...\FinRobot
AIF = ROOT / "ai_in_finance"
RUNS = ["GOOG:2025", "AMZN:2025", "MA:2025"]
MODEL = "anthropic/claude-haiku-5.5"

def run(*args):
    cmd = [sys.executable, *map(str, args)]
    print(">", " ".join(cmd[1:]))
    p = subprocess.Popen(cmd, cwd=str(ROOT), stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="")
    return p.wait()

In [ ]:
# 2) Annual Report Agent MIT Selbstkorrektur (3 Laeufe, ca. 15 Min., ca. 0.05-0.13 USD je Lauf)
run(AIF / "scripts" / "run_eval.py", "--runs", *RUNS, "--models", MODEL)

In [ ]:
# 3) Annual Report Agent OHNE Selbstkorrektur (Vergleich, ca. 15 Min.)
run(AIF / "scripts" / "run_eval.py", "--runs", *RUNS, "--models", MODEL, "--no-self-correct")

In [ ]:
# 4) Baseline: dasselbe Modell, einfacher Prompt, keine Werkzeuge und keine Daten (ca. 3 Min.)
run(AIF / "scripts" / "run_baseline.py", "--runs", *RUNS, "--model", MODEL)

In [ ]:
# 5) Pruefung des Pruefers: verfaelschte Zahlen gegen die Report-Quellen (ohne API, ca. 1 Min.)
run(AIF / "scripts" / "factcheck_fpr.py")

In [ ]:
# 6) Alle Faktenchecks in eval/eval_results.csv; Stichprobe zum manuellen Labeln
#    (eine bereits gelabelte manual_review_sample.csv wird nicht ueberschrieben)
run(AIF / "scripts" / "eval_summary.py")

In [ ]:
# 7) Forecaster auswerten (erst ab 27.11. sinnvoll: eine Woche nach den ersten Prognosen)
run(AIF / "scripts" / "score_forecasts.py")

In [ ]:
# 8) Zusammenfassung fuer den Bericht
run(AIF / "scripts" / "eval_report.py")
from IPython.display import Markdown, Image, display
display(Image(str(AIF / "eval" / "summary_chart.png")))

## Manuelle Stichprobe
`eval/manual_review_sample.csv` in Excel oeffnen und je Zeile in **manual_correct** eintragen: `1` = Zahl stimmt im Kontext (Abgleich mit 10-K bzw. Quelle), `0` = falsch.
Gruppe *found* misst, ob der Faktencheck zu grosszuegig ist; Gruppe *not found*, ob er korrekte (z. B. selbst berechnete) Zahlen verwirft. Danach Zelle 8 erneut ausfuehren.